# Further Performance Metrics

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
X_train = pd.read_csv("../data/X_train_final_smote.csv")
X_test = pd.read_csv("../data/X_test_final.csv")
X_val = pd.read_csv("../data/X_val_final.csv")

y_train = pd.read_csv("../data/y_train_final_smote.csv")
y_test = pd.read_csv("../data/y_test_final.csv")
y_val = pd.read_csv("../data/y_val_final.csv")

## 5 Combinations of Features

In [ ]:
feature_sets = {
    "set_1": X_train.columns.tolist(),

    "set_2": [
        "feature_a",
        "feature_b",
        "feature_c"
    ],

    "set_3": [
        "feature_a",
        "feature_b"
    ],

    "set_4": [
        "feature_a",
        "feature_d",
        "feature_e"
    ],

    "set_5": [
        "feature_b"
    ]
}

Create data splits.

In [ ]:
data_splits = {}

for name, features in feature_sets.items():
    data_splits[name] = {
        "X_train": X_train[features],
        "X_val": X_val[features],
        "X_test": X_test[features],
    }

In [ ]:
import warnings
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
pd.pandas.set_option("display.max_columns", None)

In [ ]:
models = [
    {
        "name": "RandomForest",
        "estimator": RandomForestClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "max_depth": [None, 8, 12],
            "min_samples_split": [2, 4, 6],
            "max_features": ['sqrt', 'log2']
        }
    },
    {
        "name": "XGBoost",
        "estimator": XGBClassifier(eval_metric='mlogloss', random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [3, 4, 5]
        }
    },
    {
        "name": "LogisticRegression",
        "estimator": LogisticRegression(random_state=42, max_iter=3000),
        "param_grid": {
            "penalty": ['l1', 'l2'],
            "C": [0.01, 0.1, 1, 10],
            "solver": ['liblinear', 'saga']
    }
}
]


## Model Training

In [ ]:
import my_tools

all_results = {}

for combo_name, data in data_splits.items():
    print(f"\n==============================")
    print(f"Running feature set: {combo_name}")
    print(f"==============================")

    results_df, best_models =  my_tools.train_ensemble_models(
        data["X_train"], y_train,
        data["X_val"], y_val,
        data["X_test"], y_test,
        models
    )

    all_results[combo_name] = results_df

In [ ]:
with open("../data/model_feature_comparison.txt", "w") as f:
    for combo_name, df in all_results.items():
        f.write("\n" + "="*60 + "\n")
        f.write(f"FEATURE SET: {combo_name}\n")
        f.write("="*60 + "\n\n")
        f.write(df.to_string(index=False))
        f.write("\n\n")